# Power control

> How much power a transmission uses, and whether it is sent at all: fixed, channel inversion, or from a prediction of the channel.

In [ ]:
#| default_exp radio.power_control

In [ ]:
#| hide
from nbdev.showdoc import *

A `PowerControl` decides the transmit power `P` from the channel gain `|h|²`, and whether to transmit at all. A
link it does not use is an *outage*: nothing is sent, so the receiver gets nothing.

* `FixedPower` uses the same power always, by default the link's `tx_power_w`.
* `ChannelInversion` uses just enough power to reach a target SNR: `P = γ σ² / |h|²`, where `γ` is the target SNR
  and `σ²` the noise power. A link needing more than `max_power` is not used.
* `MessagePower` uses the power the sender chose, e.g. a learned power control, put in the message's metadata.

`allocate` is vectorized (numpy arrays or torch tensors), so training code can handle a whole batch of links at once.
As a block, a power control gives the `power` and `active` of a step's transmissions from their `h`, calling
`allocate` for each with `|h|²` (the mean over the subcarriers). Nodes at the same place need no power.

**A broadcast is one emission, at one power.** Its copies (`group`, see `Network.broadcast`) are one signal, so they
can't have a power each. Each copy is allocated as above, then all of them get the most power any copy that would be
sent needs: enough for the weakest neighbour within the budget. The others hear the same signal at that power,
stronger than they need, or weaker than they need if they are beyond the budget. None is sent if no copy would be.
Per-link power is for unicasts: one message per link, as in C3-WM.

This is the role of `sionna.sys.power_control` in Sionna SYS (`open_loop_uplink_power_control`,
`downlink_fair_power_control`). Here a power control is a block (see `block`), and it may keep a state: what it
learned of each link, or where the nodes reported they are.

In [ ]:
#| export
from __future__ import annotations

import math
from abc import ABC, abstractmethod
from collections import deque
from typing import Callable, Sequence, Tuple

import numpy as np

from comm_core.block import Block, _full, _is_torch
from comm_core.core import Message, Transmission
from comm_core.channel.fading import effective_coefficient
from comm_core.net.topology import Link

In [ ]:
#| export
class PowerControl(Block, ABC):
    "How much power a transmission uses, and whether it is sent at all."

    outputs = ('power', 'active')

    @abstractmethod
    def allocate(
        self,
        gain, # Channel power gain |h|^2 (a number, an array or a tensor)
        noise_power, # Noise power at the receiver
        link: Link | None = None, # The link (for one transmission)
        message: Message | None = None, # The message (for one transmission)
    ):
        "(power, transmit): the transmit power, and whether to transmit (False: outage), shaped like `gain`."

    def call(
        self,
        h: np.ndarray, # h of each transmission at each subcarrier, (n, K)
        noise_power: float, # Noise power at the receivers
        link: Sequence[Link], # Their links
        transmission: Sequence[Transmission] | None = None, # The transmissions (for their messages)
        group: Sequence | None = None, # The broadcast of each transmission, None for a unicast (None: all unicasts)
    ) -> Tuple[np.ndarray, np.ndarray]:
        "(power, active) of each transmission: `allocate`, one by one, then one power per broadcast; a disabled link is not active."
        n = len(link)
        power, active = np.zeros(n), np.zeros(n, dtype=bool)
        for k in range(n):
            coefficient = effective_coefficient(h[k])
            if math.isinf(abs(coefficient)):              # the same place: a perfect link, no power needed
                active[k] = link[k].enabled
                continue
            message = transmission[k].message if transmission is not None else None
            p, ok = self.allocate(abs(coefficient) ** 2, noise_power, link=link[k], message=message)
            power[k], active[k] = float(p), bool(ok) and link[k].enabled
        emissions = {}                                     # a broadcast's copies: one signal, at one power
        for k, g in enumerate(group if group is not None else ()):
            if g is not None:
                emissions.setdefault(g, []).append(k)
        for rows in emissions.values():
            sent = [k for k in rows if active[k]]
            if not sent:                                   # no copy would be sent: the broadcast is not
                continue
            p = max(power[k] for k in sent)                # what its weakest neighbour within the budget needs
            for k in rows:
                power[k], active[k] = p, link[k].enabled and (active[k] or p > 0)
        return power, active


class FixedPower(PowerControl):
    "The same power always."

    def __init__(
        self,
        power: float | None = None, # Transmit power; None: the link's `tx_power_w` (1 without a link)
    ):
        self.power = power

    def allocate(self, gain, noise_power, link=None, message=None):
        p = self.power if self.power is not None else (link.tx_power_w if link is not None else 1.0)
        return _full(gain, p), _full(gain, True, boolean=True)


class ChannelInversion(PowerControl):
    "Just enough power to reach `target_snr_db`: P = snr noise_power / |h|^2; links needing more than `max_power` are not used."

    def __init__(
        self,
        target_snr_db: float = 10.0, # SNR at the receiver
        max_power: float = math.inf, # Power budget
    ):
        self.target_snr_db, self.max_power = target_snr_db, max_power

    def allocate(self, gain, noise_power, link=None, message=None):
        if not (_is_torch(gain) or isinstance(gain, np.ndarray)):
            gain = np.float64(gain)
        with np.errstate(divide='ignore'):                 # |h|^2 = 0: infinite power, an outage
            power = 10 ** (self.target_snr_db / 10) * noise_power / gain
        return power, (power <= self.max_power) & (abs(power) < math.inf)   # infinite power: never sent


class MessagePower(PowerControl):
    "The power the sender chose (e.g. a learned power control), in `message.metadata[key]`; no power or 0: not sent."

    def __init__(
        self,
        key: str = 'tx_power', # Metadata key of the power
        max_power: float = math.inf, # Larger powers are clipped to it
    ):
        self.key, self.max_power = key, max_power

    def allocate(self, gain, noise_power, link=None, message=None):
        if message is None:
            raise ValueError("MessagePower reads the power from the message")
        p = min(float(message.metadata.get(self.key, 0.0)), self.max_power)
        return p, p > 0

In [ ]:
show_doc(PowerControl.allocate)

In [ ]:
show_doc(PowerControl.call)

In [ ]:
p, ok = ChannelInversion(target_snr_db=10, max_power=10).allocate(np.array([1.0, 0.5, 0.01, 0.0]), noise_power=1.0)
assert np.allclose(p[:3], [10, 20, 1000]) and p[3] == np.inf and ok.tolist() == [True, False, False, False]
# as a block: one row per transmission; a disabled link is not sent, nodes at the same place need no power
links = [Link('a', 'b'), Link('a', 'c', enabled=False), Link('a', 'd')]
power, active = ChannelInversion(10)(np.array([[0.5], [1.0], [math.inf]]), 1.0, links)     # |h|^2 = 0.25: 40
assert power.tolist() == [40.0, 10.0, 0.0] and active.tolist() == [True, False, True]
p, ok = FixedPower().allocate(0.3, 1.0, link=Link('a', 'b', tx_power_w=0.2))
assert float(p) == 0.2 and bool(ok)
assert not ChannelInversion(10).allocate(0.0, 1.0)[1]                      # no channel: not sent, even without a budget
assert MessagePower(max_power=5).allocate(1.0, 1.0, message=Message('a', 'b', None, metadata={'tx_power': 9})) == (5.0, True)
assert MessagePower().allocate(1.0, 1.0, message=Message('a', 'b', None)) == (0.0, False)

In [ ]:
# a broadcast is one emission: its copies get the power of the weakest one within the budget, even those beyond it
links = [Link('a', n) for n in 'bcde']
h = np.sqrt([[1.0], [0.25], [0.001], [0.25]])                              # needs 10, 40, 10_000 (> 100), 40
power, active = ChannelInversion(10, max_power=100)(h, 1.0, links, group=['x', 'x', 'x', None])
assert power.tolist() == [40.0, 40.0, 40.0, 40.0] and active.all()
power, active = ChannelInversion(10, max_power=100)(h[:3], 1.0, links[:3], group=['x', 'x', None])
assert np.allclose(power, [40, 40, 10_000]) and active.tolist() == [True, True, False]   # a unicast keeps its own
power, active = ChannelInversion(10, max_power=1)(h[:2], 1.0, links[:2], group=['x', 'x'])
assert not active.any()                                                    # no copy within the budget: not sent
power, active = FixedPower(2.0)(h[:2], 1.0, [links[0], Link('a', 'c', enabled=False)], group=['x', 'x'])
assert active.tolist() == [True, False]                                    # a disabled link still gets nothing

### Without the channel at the transmitter

Channel inversion needs the current channel gain `|h|²`, which a transmitter rarely has: it would take feedback
from the receiver, and the channel changes in the meantime. `PredictedChannelInversion` predicts it instead, from
the link's past gains (as in C3-WM):

1. **Prediction:** `ŷ = max(0, Ψ([|h_{t-1}|², ..., |h_{t-L}|²]))`, where `Ψ` is a predictor, e.g. learned.
2. **Power:** `ρ = Γ σ² / ŷ`, where `Γ` is the target SNR.
3. **Scheduling:** the transmission is sent only if `ρ ≤ ρ_max`.
4. **Start of a link:** until the link has `L` past gains, it sends at a fallback power.

The gain of each transmission is added to its link's history afterwards (the channel known after the fact, e.g.
from the receiver's feedback). The history is per directed link, one entry per transmission, and is forgotten when
the channel is reset (a new episode).

The SNR at the receiver is then `ρ |h_t|² / σ²`, which misses the target by the prediction's error.

`gain_windows` turns gain sequences into a predictor's training set: past `L` gains → the next one. Collect the
sequences by running the environment and reading the fading's gains, e.g. `RayTracedFading.path_gain`.
`fit_gain_predictor` fits a linear `Ψ` by regularized least squares.

**Fit in dB or on `|h|²`.** Gains typically span tens of dB. A least-squares fit of `|h|²` itself
(`domain='linear'`, C3-WM's eq. 24) is dominated by the strongest links, and overestimates the weak ones: a bias a
margin must make up for. A fit in dB (`domain='db'`) is unbiased. Neither makes the target reliable when the past
gains say little about the next one, e.g. when nodes move many wavelengths between transmissions. A map of the
channel does better there (see `MapChannelInversion` below, and the example
[power control without channel knowledge](../examples/06_predicted_power.ipynb)).

**Margin.** Even an unbiased prediction misses half the time, by its error. `margin_db` aims that much above the
target, trading power for reliability.

In [ ]:
#| export
def gain_windows(
    gains: Sequence[float], # One link's channel gains |h|^2, in time order
    history: int, # Past gains per window (L)
) -> Tuple[np.ndarray, np.ndarray]:
    "(X, y): each gain y_t with the `history` gains before it, most recent first: X_t = [g_{t-1}, ..., g_{t-L}]."
    g = np.asarray(gains, float)
    if len(g) <= history:
        return np.zeros((0, history)), np.zeros(0)
    X = np.stack([g[t - history:t][::-1] for t in range(history, len(g))])
    return X, g[history:]


def fit_gain_predictor(
    sequences: Sequence[Sequence[float]], # Gain sequences |h|^2 (e.g. one per link and episode)
    history: int, # Past gains the predictor sees (L)
    ridge: float = 1e-6, # L2 regularization of the weights
    domain: str = 'linear', # Fit |h|^2 itself ('linear'), or its value in dB ('db', for power inversion; see below)
) -> Callable:
    "A linear predictor of the next channel gain from the last `history` ones (least squares with a bias)."
    if domain not in ('linear', 'db'):
        raise ValueError(f"domain: 'linear' or 'db', not {domain!r}")
    pairs = [gain_windows(s, history) for s in sequences]
    X = np.concatenate([x for x, _ in pairs])
    y = np.concatenate([t for _, t in pairs])
    if not len(y):
        raise ValueError(f"no sequence is longer than the history ({history})")
    if domain == 'db':
        to = lambda g: 10 * np.log10(np.maximum(np.asarray(g, float), 1e-30))
        back = lambda v: 10 ** (v / 10)
    else:
        scale = float(np.mean(y)) or 1.0                   # gains are tiny (e.g. 1e-9): fit in units of their mean
        to = lambda g: np.asarray(g, float) / scale
        back = lambda v: v * scale
    A = np.c_[to(X), np.ones(len(X))]
    w = np.linalg.solve(A.T @ A + ridge * len(A) * np.eye(A.shape[1]), A.T @ to(y))

    def predictor(past):
        return float(back(np.r_[to(past), 1.0] @ w))

    predictor.weights, predictor.domain = w, domain
    return predictor


class PredictedChannelInversion(PowerControl):
    "Channel inversion with a predicted gain: |h|^2 predicted from the link's past gains, as without channel knowledge."

    def __init__(
        self,
        predictor: Callable, # Past gains (most recent first, `history` of them) -> the current gain, e.g. `fit_gain_predictor`
        history: int, # Past gains the predictor needs (L)
        target_snr_db: float = 10.0, # Target SNR at the receiver (Γ)
        max_power: float = math.inf, # Power budget (ρ_max); more: not sent
        fallback_power: float | None = None, # Power while a link has fewer than `history` past gains; None: the link's tx_power_w
        margin_db: float = 0.0, # Extra SNR aimed for, against prediction errors
    ):
        self.predictor, self.history, self.target_snr_db, self.margin_db = predictor, history, target_snr_db, margin_db
        self.max_power, self.fallback_power = max_power, fallback_power
        self._past = {}
        self.predicted = None                              # the last prediction (None: fallback power)

    def reset(self, seed=None):
        self._past.clear()
        self.predicted = None

    def allocate(self, gain, noise_power, link=None, message=None):
        if link is None:
            raise ValueError("PredictedChannelInversion keeps a history per link: give the link")
        past = self._past.setdefault((link.sender, link.receiver), deque(maxlen=self.history))
        if len(past) < self.history:
            self.predicted = None
            power = self.fallback_power if self.fallback_power is not None else link.tx_power_w
        else:
            self.predicted = max(0.0, float(self.predictor(np.array(past)[::-1])))
            snr = 10 ** ((self.target_snr_db + self.margin_db) / 10)
            power = snr * noise_power / self.predicted if self.predicted > 0 else math.inf
        past.append(float(gain))                           # known after the transmission
        return power, math.isfinite(power) and power <= self.max_power

In [ ]:
show_doc(gain_windows)

In [ ]:
show_doc(fit_gain_predictor)

In [ ]:
show_doc(PredictedChannelInversion)

In [ ]:
X, y = gain_windows([1, 2, 3, 4, 5], history=2)
assert X.tolist() == [[2, 1], [3, 2], [4, 3]] and y.tolist() == [3, 4, 5]      # most recent first
assert gain_windows([1, 2], history=2)[0].shape == (0, 2)
# a predictable channel: an AR(1) process around 1e-9; the fit recovers its coefficient
rng = np.random.default_rng(0)
def ar1(n, a=0.9, mean=1e-9):
    g = [mean]
    for _ in range(n - 1):
        g.append(mean + a * (g[-1] - mean) + 0.1 * mean * rng.standard_normal())
    return np.abs(g)
psi = fit_gain_predictor([ar1(500) for _ in range(20)], history=3)
assert abs(psi.weights[0] - 0.9) < 0.05 and abs(psi.weights[1]) < 0.05
assert abs(psi([1.2e-9, 1.1e-9, 1.0e-9]) - (1e-9 + 0.9 * 0.2e-9)) < 0.02e-9
# in dB: a gain halving every step (-3 dB per step) is predicted exactly
halving = [1e-6 * 0.5 ** t for t in range(30)]
psi_db = fit_gain_predictor([halving], history=2, domain='db', ridge=0)
assert abs(psi_db([halving[5], halving[4]]) / halving[6] - 1) < 1e-6 and psi_db.domain == 'db'
try:
    fit_gain_predictor([halving], history=2, domain='log2'); raise AssertionError('no error')
except ValueError:
    pass

In [ ]:
# fallback while the history is short, then the prediction; one history per directed link; reset forgets
const = lambda past: past[0]                                     # predict "the same as last time"
pc = PredictedChannelInversion(const, history=2, target_snr_db=10, max_power=1e3, fallback_power=0.5)
ab, ba = Link('a', 'b'), Link('b', 'a')
assert pc.allocate(0.1, 1.0, link=ab) == (0.5, True) and pc.predicted is None
assert pc.allocate(0.2, 1.0, link=ab) == (0.5, True)
p, ok = pc.allocate(0.4, 1.0, link=ab)                           # predicts 0.2 (the last gain), not the true 0.4
assert abs(p - 10 / 0.2) < 1e-9 and ok and pc.predicted == 0.2
assert pc.allocate(1.0, 1.0, link=ba) == (0.5, True)             # the other direction has its own history
p, ok = pc.allocate(1.0, 1.0, link=ab)                           # predicts 0.4
assert abs(p - 25) < 1e-9
pc.margin_db = 3.0
assert abs(pc.allocate(1.0, 1.0, link=ab)[0] - 10 ** 1.3 / 1.0) < 1e-9   # 3 dB more
pc_low = PredictedChannelInversion(lambda past: 1e-6, history=1, target_snr_db=10, max_power=1.0)
pc_low.allocate(1.0, 1.0, link=ab)
assert pc_low.allocate(1.0, 1.0, link=ab) == (1e7, False)       # the predicted channel needs too much power
assert PredictedChannelInversion(lambda past: -1.0, history=1).allocate(1.0, 1.0, link=ab)[1]  # fallback first
neg = PredictedChannelInversion(lambda past: -1.0, history=1)
neg.allocate(1.0, 1.0, link=ab)
assert neg.allocate(1.0, 1.0, link=ab) == (math.inf, False)      # negative predictions are clipped to 0: not sent

### With a map and reported positions

The history of a link's gains says little about the next gain when the nodes move far between transmissions (e.g.
a whole grid cell, many wavelengths). A ray-traced map of the area says more, if the transmitter knows where both
nodes are:

* **Its own position:** the transmitter knows it.
* **The receiver's:** it is known as of its last report, e.g. its cell and heading sent along with its last message
  (a few bits).

`MapChannelInversion` looks the map up at the transmitter's position and at the cells the receiver can have reached
since its report (`reachable`), and takes the weakest of those gains: a prediction that is robust to where the
receiver went. With a fallback, e.g. `PredictedChannelInversion`, it covers the time before a receiver's first
report.

The reports are taken from the map's node positions, one round late: at every round (`update`), the positions of
the previous round become the reported ones. `states` adds what `reachable` needs, e.g. headings, read at every
round and reported the same way.

**Results.** Compared by the power each needs to reach Γ on a share of the transmissions perfect CSI can serve
(the [power control example](../examples/06_predicted_power.ipynb) and stable-marl's C3-WM tutorial):

* **History predictors** (the last gain, or a linear fit of the last 4) reach Γ 90% of the time, but not 95%:
  their errors spread too widely.
* **The robust map lookup** with the receiver's heading reached 95% of them at 0.8× perfect CSI's mean power on
  FindGoal's ray-traced channel, and 99% at 1.4× in a synthetic room. C3-WM's predictor needed 2.4× for 95%.
* **Without the heading,** the receiver can be in any neighbouring cell, and the weakest of them is often far too
  pessimistic.


In [ ]:
#| export
class MapChannelInversion(PowerControl):
    "Channel inversion with the gain looked up in a map: the weakest gain over the cells the receiver can have reached since its last report."

    def __init__(
        self,
        map, # The channel map: `gain_between(tx_position, rx_position)` and `node_positions` (e.g. `RayTracedFading`)
        reachable: Callable | None = None, # reachable(position, state) -> positions the receiver can be at now; None: its reported one
        states: Callable | None = None, # states() -> {node: state} (e.g. headings), read at every round
        target_snr_db: float = 10.0, # Target SNR at the receiver (Γ)
        max_power: float = math.inf, # Power budget (ρ_max); more: not sent
        margin_db: float = 0.0, # Extra SNR aimed for, against the map's errors
        fallback: PowerControl | None = None, # Before a receiver's first report; None: the link's tx_power_w
    ):
        self.map, self.reachable, self.states = map, reachable, states
        self.target_snr_db, self.max_power, self.margin_db, self.fallback = target_snr_db, max_power, margin_db, fallback
        self.reset()

    def reset(self, seed=None):
        self._now, self._reported, self._now_states, self._reported_states = {}, {}, {}, {}
        self.predicted = None
        if self.fallback is not None:
            self.fallback.reset(seed)

    def update(self, time):
        "A new round: what the nodes reported last round becomes known; note where they are now."
        self._reported, self._reported_states = self._now, self._now_states
        self._now = dict(self.map.node_positions)
        self._now_states = dict(self.states()) if self.states is not None else {}
        if self.fallback is not None:
            self.fallback.update(time)

    def allocate(self, gain, noise_power, link=None, message=None):
        if link is None:
            raise ValueError("MapChannelInversion needs the link")
        tx = self.map.node_positions.get(link.sender)
        rx = self._reported.get(link.receiver)
        if tx is None or rx is None:                       # no report yet: the fallback
            self.predicted = None
            if self.fallback is not None:
                return self.fallback.allocate(gain, noise_power, link=link, message=message)
            return link.tx_power_w, link.tx_power_w <= self.max_power
        if self.fallback is not None:                       # keeps the fallback's own state (e.g. gain history) current
            self.fallback.allocate(gain, noise_power, link=link, message=message)
        cells = [rx] if self.reachable is None else list(self.reachable(rx, self._reported_states.get(link.receiver)))
        here = [c for c in cells if tuple(c) == tuple(tx)]
        cells = [c for c in cells if tuple(c) != tuple(tx)] or here    # the receiver is elsewhere, if it can be
        gains = []
        for c in cells:
            try:
                gains.append(self.map.gain_between(tx, c))
            except KeyError:                                # a cell outside the map: not a place to be
                pass
        if not gains:
            gains = [self.map.gain_between(tx, rx)]
        self.predicted = min(gains)
        if math.isinf(self.predicted):                      # the receiver may be right here: no prediction needed
            return 0.0, True
        snr = 10 ** ((self.target_snr_db + self.margin_db) / 10)
        power = snr * noise_power / self.predicted if self.predicted > 0 else math.inf
        return power, math.isfinite(power) and power <= self.max_power

In [ ]:
show_doc(MapChannelInversion)

In [ ]:
from comm_core.channel.ray_tracing import RayTracedFading

In [ ]:
# a corridor of 5 cells; the gain falls 10 dB per cell
cells_1d = [(x, 0) for x in range(5)]
G1 = np.array([[0 if r == t else 10 ** (-(5 + 1 * abs(r - t))) for t in range(5)] for r in range(5)])
corridor = RayTracedFading(np.sqrt(G1), cells_1d)
step_right = lambda pos, heading: [pos, (pos[0] + heading, pos[1])]     # stay, or one cell in the heading
heading = {'a': 1, 'b': 1}
mc = MapChannelInversion(corridor, reachable=step_right, states=lambda: heading, target_snr_db=10, max_power=1e6,
                         fallback=FixedPower(0.5))
ab = Link('a', 'b')
corridor.set_positions({'a': (0, 0), 'b': (1, 0)}); mc.update(1.0)
assert mc.allocate(1.0, 1.0, link=ab) == (0.5, True) and mc.predicted is None     # nothing reported yet: the fallback
corridor.set_positions({'a': (0, 0), 'b': (2, 0)}); mc.update(2.0)                # b moved; its report says (1, 0)
p, ok = mc.allocate(1.0, 1.0, link=ab)
assert mc.predicted == corridor.gain_between((0, 0), (2, 0)) and abs(p - 10 / 1e-7) < 1e-3   # the weaker of (1, 0), (2, 0)
heading['b'] = -1
corridor.set_positions({'a': (0, 0), 'b': (2, 0)}); mc.update(3.0)                # reported: (2, 0), heading right
corridor.set_positions({'a': (0, 0), 'b': (1, 0)}); mc.update(4.0)                # reported: (2, 0), heading left
mc.allocate(1.0, 1.0, link=ab)
assert mc.predicted == corridor.gain_between((0, 0), (2, 0))                       # (1, 0) is stronger: (2, 0) counts
corridor.set_positions({'a': (1, 0), 'b': (1, 0)}); mc.update(5.0)                # a moved into b's reported cell
mc.allocate(1.0, 1.0, link=ab)
assert mc.predicted == corridor.gain_between((1, 0), (0, 0))                       # b can only be at (0, 0) then
mc.reset()
assert mc.allocate(1.0, 1.0, link=ab) == (0.5, True)

### In a channel

A channel resets its power control with the network (a new episode), and passes it every round (`update`). Its
pipeline gives the power control each transmission's broadcast (`group`):

In [ ]:
from comm_core import AnalyticalBackend, FullMeshTopology, IdealProtocol, Network, Node, SystemLevelChannel

In [ ]:
const = lambda past: past[0]
ab = Link('a', 'b')
# the channel resets it with the network (a new episode)
ch = SystemLevelChannel(power_control=PredictedChannelInversion(const, history=1, fallback_power=0.5))
t = Transmission(Message('a', 'b', None, size_bits=8), 'a', 'b', 0.0)
ch.transmit(t, ab); ch.transmit(t, ab)
assert ch.power_control.predicted == 1.0
ch.reset(seed=0)
assert ch.power_control.predicted is None and ch.transmit(t, ab).metadata['tx_power'] == 0.5
# rounds reach the power control: a static receiver is predicted exactly from the map
static = RayTracedFading(np.sqrt(G1), cells_1d, node_positions={'a': (0, 0), 'b': (2, 0)})
radio = SystemLevelChannel(fading=static, power_control=MapChannelInversion(static, target_snr_db=10), noise_power=1e-9)
net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(radio))
snrs = []
for _ in range(3):
    net.send(Message('a', 'b', None, size_bits=64))
    snrs += [r.snr_db for r in net.step()]
assert snrs[0] != 10 and all(abs(s - 10) < 1e-9 for s in snrs[1:])     # first round: the link's own power; then exact

In [ ]:
# a broadcast through a network: both neighbours hear one signal, at the power the farther one needs
static = RayTracedFading(np.sqrt(G1), cells_1d, node_positions={'a': (0, 0), 'b': (1, 0), 'c': (3, 0)})
radio = SystemLevelChannel(fading=static, power_control=ChannelInversion(target_snr_db=10), noise_power=1e-9)
net = Network([Node(n) for n in 'abc'], FullMeshTopology(list('abc')), IdealProtocol(), AnalyticalBackend(radio))
net.broadcast('a', payload=1, size_bits=64)
results = {r.transmission.receiver: r for r in net.step()}
assert results['b'].metadata['tx_power'] == results['c'].metadata['tx_power'] == 1.0     # |h|^2 = 1e-8 at c
assert abs(results['c'].snr_db - 10) < 1e-9 and abs(results['b'].snr_db - 30) < 1e-9    # b: 20 dB more than it needs
assert abs(net.metrics.summary().total_energy - 64) < 1e-9                               # charged once: P x 64 symbols

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()